# scRNA-seq Analysis Pipeline: CLL Paired Samples (P457 & P477)

**Study:** AID-positive CLL — diagnosis vs. refractory, peripheral blood  
**Data:** 2 multiplexed `.h5` files (chips MV1 & MV2), 4 biological samples total  
**Goal:** Cell annotation, tumor cell identification, pathway enrichment  
**Author:** Julieta Sepúlveda-Yáñez, PhD

---

## Study design

| Patient | Time point | Chip |
|---------|------------|------|
| P457 (S457) | Diagnosis | MV1 |
| P457 (S457) | Refractory | MV2 |
| P477 (S477) | Diagnosis | MV2 |
| P477 (S477) | Refractory | MV1 |

Each `.h5` file is **multiplexed** — two biological samples captured together on the same chip.  
The demultiplexing table (your labeling file) assigns each cell barcode to its correct patient, time point, and cell type.

---

> **Kernel required:** R kernel (IRkernel).  
> Install in R with: `install.packages("IRkernel"); IRkernel::installspec()`  
> Then restart VS Code and select the **R** kernel for this notebook.

## Section 0 — Package Installation

> Run this cell **once**. It installs all required packages. Safe to skip on subsequent runs if packages are already installed.

In [14]:
# ── Bioconductor packages ──────────────────────────────────────────────────
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

BiocManager::install(c(
  "SingleR",       # Automated cell type annotation
  "celldex",       # Reference datasets for SingleR
  "scran",         # Pooling-based size factor normalization
  "edgeR",         # Used internally by scran / DESeq2
  "DESeq2",        # Pseudobulk differential expression
  "fgsea",         # Fast gene set enrichment analysis
  "msigdbr",       # MSigDB gene sets (Hallmark, KEGG, etc.)
  "BiocParallel"   # Parallel processing
), update = FALSE, ask = FALSE)

# ── CRAN packages ──────────────────────────────────────────────────────────
install.packages(c(
  "Seurat",        # Core scRNA-seq toolkit
  "SeuratObject",
  "harmony",       # Batch correction
  "tidyverse",     # Data wrangling
  "ggplot2",       # Plotting
  "ggrepel",       # Non-overlapping labels on plots
  "patchwork",     # Combine ggplots
  "hdf5r",         # Read .h5 files
  "viridis",       # Color palettes
  "pheatmap",      # Heatmaps
  "RColorBrewer"   # Color palettes
), repos = "https://cloud.r-project.org")

message("All packages installed.")

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.1 (2026-06-24)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'SingleR' 'scran' 'edgeR' 'DESeq2' 'fgsea'
  'msigdbr' 'BiocParallel'”
Installing package(s) 'celldex'

also installing the dependency ‘alabaster.base’


Package which is only available in source form, and may need
  compilation of C/C++/Fortran: ‘alabaster.base’

installing the source packages ‘alabaster.base’, ‘celldex’


Warning message in install.packages(...):
“installation of package ‘alabaster.base’ had non-zero exit status”
Warning message in install.packages(...):
“installation of package ‘celldex’ had non-zero exit status”



The downloaded binary packages are in
	/var/folders/jy/38s27vtx02l6xzwx6nhwdz4w0000gn/T//RtmpftcN05/downloaded_packages


All packages installed.



## Section 1 — Load Libraries

In [13]:
library(Seurat)
library(harmony)
library(SingleR)
library(celldex)
library(scran)
library(DESeq2)
library(fgsea)
library(msigdbr)
library(tidyverse)
library(ggplot2)
library(ggrepel)
library(patchwork)
library(hdf5r)
library(pheatmap)
library(RColorBrewer)
library(viridis)
library(BiocParallel)
library(dplyr)

set.seed(42)
register(MulticoreParam(4))  # Adjust to available cores

message("Libraries loaded.")

ERROR: Error in library(celldex): there is no package called ‘celldex’


## Section 2 — Load Multiplexed `.h5` Files + Demultiplexing

### What happens here

Each `.h5` file was produced by 10x Cell Ranger from a chip that contains **two patients mixed together**.  
We already have a demultiplexing table that maps each barcode to its correct `subject`, `status`, and `celltype`.

**Barcode format in the demux file:** `MV1:AAACCTGCACAACTGT-1`  
**Barcode format in the `.h5` matrix:** `AAACCTGCACAACTGT-1` (no chip prefix)

The strategy is:
1. Load the full count matrix from each `.h5` file
2. Strip the chip prefix from the demux barcodes to match the matrix
3. Keep only barcodes present in **both** the matrix and the demux table
4. Build one Seurat object per chip and attach all metadata

> **Update the three file path variables** in the cell below before running.

In [16]:
# ── UPDATE THESE PATHS ────────────────────────────────────────────────────
file_MV1   <- "data/sample_p1_feature_bc_matrix.h5"   # Chip MV1: P457-Dx + P477-Ref
file_MV2   <- "data/sample_p2_feature_bc_matrix.h5"   # Chip MV2: P477-Dx + P457-Ref
file_demux <- "data/MV12_labeling.csv"                   # Your demultiplexing table
# ──────────────────────────────────────────────────────────────────────────

# Step 2.1 — Load demultiplexing table
# Expected columns: source, cell, subject, status, celltype
demux <- read.table(file_demux,
                    header = TRUE,
                    sep    = ",",
                    stringsAsFactors = FALSE)

cat("Demux table preview:\n")
print(head(demux))
cat("\nCells per chip x subject:\n")
print(table(demux$source, demux$subject))

Demux table preview:
  source                   cell subject     status celltype
1    MV1 MV1:AAACCTGCACAACTGT-1    S477 refractory TcellCD8
2    MV1 MV1:AAACCTGCACAGTCGC-1    S477 refractory TcellCD4
3    MV1 MV1:AAACCTGCACCTTGTC-1    S477 refractory TcellCD8
4    MV1 MV1:AAACCTGCAGTCACTA-1    S477 refractory TcellCD8
5    MV1 MV1:AAACCTGCATCTCCCA-1    S457  diagnosis    Bcell
6    MV1 MV1:AAACCTGGTACCGAGA-1    S477 refractory TcellCD8

Cells per chip x subject:
     
      S457 S477
  MV1 3522 4192
  MV2 1255 6224


In [17]:
# Step 2.2 — Add structured metadata columns
# Creates patient, timepoint, and sample_id from the demux subject/status fields

demux <- demux %>%
  mutate(sample_id = paste(subject, status, sep = "_"))

cat("Cells per biological sample (should show 4 groups):\n")
print(table(demux$sample_id))

Cells per biological sample (should show 4 groups):

 S457_diagnosis S457_refractory  S477_diagnosis S477_refractory 
           3522            1255            6224            4192 


In [18]:
# Step 2.3 — Load and filter chip MV1

message("Loading MV1...")
counts_MV1 <- Read10X_h5(file_MV1)
if (is.list(counts_MV1)) counts_MV1 <- counts_MV1[["Gene Expression"]]

demux_MV1 <- demux %>%
  filter(source == "MV1") %>%
  mutate(barcode_raw = sub("^MV1:", "", cell))   # Strip "MV1:" prefix

valid_MV1 <- intersect(demux_MV1$barcode_raw, colnames(counts_MV1))
message("MV1: ", length(valid_MV1), " matched barcodes out of ",
        ncol(counts_MV1), " in matrix | ", nrow(demux_MV1), " in demux")

obj_MV1 <- CreateSeuratObject(
  counts       = counts_MV1[, valid_MV1],
  project      = "MV1",
  min.cells    = 3,
  min.features = 200
)
obj_MV1$chip <- "MV1" 

Loading MV1...

MV1: 7714 matched barcodes out of 8099 in matrix | 7714 in demux



In [19]:
# Step 2.4 — Load and filter chip MV2

message("Loading MV2...")
counts_MV2 <- Read10X_h5(file_MV2)
if (is.list(counts_MV2)) counts_MV2 <- counts_MV2[["Gene Expression"]]

demux_MV2 <- demux %>%
  filter(source == "MV2") %>%
  mutate(barcode_raw = sub("^MV2:", "", cell))

valid_MV2 <- intersect(demux_MV2$barcode_raw, colnames(counts_MV2))
message("MV2: ", length(valid_MV2), " matched barcodes out of ",
        ncol(counts_MV2), " in matrix | ", nrow(demux_MV2), " in demux")

obj_MV2 <- CreateSeuratObject(
  counts       = counts_MV2[, valid_MV2],
  project      = "MV2",
  min.cells    = 3,
  min.features = 200
)
obj_MV2$chip <- "MV2" 

Loading MV2...

MV2: 7479 matched barcodes out of 7622 in matrix | 7479 in demux



In [ ]:
# Step 2.5 — Transfer demux metadata into each Seurat object

add_demux_metadata <- function(obj, demux_df) {
  demux_indexed <- demux_df
  rownames(demux_indexed) <- demux_indexed$barcode_raw
  demux_indexed$barcode_raw <- NULL
  matched       <- demux_indexed[colnames(obj), ]
  
  obj$subject        <- matched$subject
  obj$sample_id      <- matched$sample_id
  obj$celltype_demux <- matched$celltype   # Pre-existing annotation
  
  n_missing <- sum(is.na(obj$sample_id))
  if (n_missing > 0) {
    warning(n_missing, " cells without demux match — removing")
    obj <- subset(obj, subset = !is.na(sample_id))
  }
  return(obj)
}

obj_MV1 <- add_demux_metadata(obj_MV1, demux_MV1)
obj_MV2 <- add_demux_metadata(obj_MV2, demux_MV2)

cat("\nMV1 sample breakdown (should be P457_Diagnosis + P477_Refractory):\n")
print(table(obj_MV1$sample_id))
cat("\nMV2 sample breakdown (should be P477_Diagnosis + P457_Refractory):\n")
print(table(obj_MV2$sample_id))


ERROR: Error in column_to_rownames(., "barcode_raw"): could not find function "column_to_rownames"


## Section 3 — Quality Control

QC is done at the **chip level** (matching the `.h5` structure), then visualized **per sample_id** so you can see whether the two biological samples within each chip have similar quality distributions.

### Metrics
| Metric | Low value | High value |
|--------|-----------|------------|
| `nFeature_RNA` | Empty droplet | Doublet |
| `nCount_RNA` | Low-quality | Doublet |
| `percent.mt` | — | Dying / lysed cell |

> **CLL-specific note:** Tumor B cells have naturally lower gene complexity than T cells. Do not set `nFeature_RNA` max too low or you will lose the non-tumor immune compartment.

In [ ]:
# Calculate QC metrics for both chips
for (chip_name in c("MV1", "MV2")) {
  obj <- get(paste0("obj_", chip_name))
  obj[["percent.mt"]]   <- PercentageFeatureSet(obj, pattern = "^MT-")
  obj[["percent.ribo"]] <- PercentageFeatureSet(obj, pattern = "^RP[SL]")
  assign(paste0("obj_", chip_name), obj)
}

message("QC metrics computed.")

In [ ]:
# Violin plots — MV1
# Colored by sample_id: each chip contains 2 biological samples
VlnPlot(obj_MV1,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "sample_id",
        ncol = 3, pt.size = 0) +
  plot_annotation(title = "QC Chip MV1 (P457-Dx + P477-Ref)")

In [ ]:
# Violin plots — MV2
VlnPlot(obj_MV2,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "sample_id",
        ncol = 3, pt.size = 0) +
  plot_annotation(title = "QC Chip MV2 (P477-Dx + P457-Ref)")

In [ ]:
# Scatter: UMIs vs genes — useful to spot doublet clouds
p1 <- FeatureScatter(obj_MV1, "nCount_RNA", "nFeature_RNA",
                     group.by = "sample_id") + ggtitle("MV1")
p2 <- FeatureScatter(obj_MV2, "nCount_RNA", "nFeature_RNA",
                     group.by = "sample_id") + ggtitle("MV2")
p1 | p2

### Step 3c — Understanding QC Thresholds and Filtering

Before we filter, we need to **count how many cells we start with** and understand what each threshold means.  
This step is one of the most important decisions in your analysis — removing too few cells leaves noise in, removing too many loses real biology.

---

#### What are we filtering?

Each row in your data matrix is a **cell barcode**. Not all barcodes correspond to real, healthy cells:

| Problem | What you see in the data | How to detect it |
|---------|--------------------------|------------------|
| **Empty droplet** | Very few genes detected (<200) | Low `nFeature_RNA` |
| **Dying / lysed cell** | High mitochondrial fraction | High `percent.mt` |
| **Doublet** (two cells captured together) | Abnormally high gene count | High `nFeature_RNA` AND `nCount_RNA` |
| **Damaged B cell** | Moderate mito, low complexity | Context-dependent |

---

#### Threshold reference values for CLL blood

The values below are based on published CLL scRNA-seq studies and general best practices:

| Parameter | Lower cutoff | Upper cutoff | Rationale |
|-----------|-------------|--------------|-----------|
| `nFeature_RNA` | **200** | **6,000** | <200 = empty droplet (Luecken & Theis 2019); >6000 likely doublet — but CLL blood can have high T cell complexity, so 6000 is safer than 5000 |
| `nCount_RNA` | *(none)* | **30,000** | Very high UMI count without proportionally more genes = doublet signal |
| `percent.mt` | *(none)* | **15%** | >15% indicates cytoplasmic RNA leakage from lysed cells (Ilicic et al. 2016); some studies use 10% for stricter filtering |

> **Key references:**  
> - Luecken MD & Theis FJ (2019). *Current best practices in single-cell RNA-seq analysis: a tutorial.* Mol Syst Biol. [doi:10.15252/msb.20188746](https://doi.org/10.15252/msb.20188746)  
> - Ilicic T et al. (2016). *Classification of low quality cells from single-cell RNA-seq data.* Genome Biol. [doi:10.1186/s13059-016-0888-1](https://doi.org/10.1186/s13059-016-0888-1)  
> - Hao Y et al. (2021). *Integrated analysis of multimodal single-cell data (Seurat v4).* Cell. [doi:10.1016/j.cell.2021.04.048](https://doi.org/10.1016/j.cell.2021.04.048)

---

#### CLL-specific considerations

- The **CLL tumor clone** (CD19⁺ CD5⁺) dominates the sample (~80–96% of cells in your patients). These cells have **lower gene complexity** than T or NK cells because they are mature, quiescent B cells with a restricted transcriptome.
- This means the `nFeature_RNA` distribution will be **bimodal**: a large peak from CLL cells (often 500–2,000 genes) and a smaller peak from T/NK/monocytes (1,500–4,000 genes). Do NOT set the upper threshold below 4,000 or you will lose all T and NK cells.
- The `percent.mt` cutoff of 15% is standard. For very high-quality CLL samples some labs use 10%, but 15% is safer to avoid losing real cells.

> **Practical rule:** Look at your violin plots from the previous cells. The thresholds should fall **in the gaps** between real cells and outliers, not cutting through the main distribution.


In [ ]:
# ── Pre-filter: how many cells do we START with? ─────────────────────────
cat("=== BEFORE FILTERING ===\n\n")

cat("Chip MV1 — total cells: ", ncol(obj_MV1), "\n")
cat("  Per sample_id:\n")
print(table(obj_MV1$sample_id))

cat("\nChip MV2 — total cells: ", ncol(obj_MV2), "\n")
cat("  Per sample_id:\n")
print(table(obj_MV2$sample_id))

cat("\nCombined (before filter): ", ncol(obj_MV1) + ncol(obj_MV2), " cells\n")

# ── Per-metric summaries to help you calibrate thresholds ────────────────
cat("\n=== METRIC SUMMARIES (before filtering) ===\n")

for (chip_name in c("MV1", "MV2")) {
  obj <- get(paste0("obj_", chip_name))
  cat("\n-- Chip", chip_name, "--\n")
  
  meta <- obj@meta.data
  for (metric in c("nFeature_RNA", "nCount_RNA", "percent.mt")) {
    x <- meta[[metric]]
    cat(sprintf("  %-15s  min=%6.0f  Q1=%6.0f  median=%6.0f  Q3=%6.0f  max=%6.0f\n",
                metric, min(x), quantile(x,.25), median(x), quantile(x,.75), max(x)))
  }
  
  # Flag potential issues
  cat("  Cells with nFeature_RNA < 200  :", sum(meta$nFeature_RNA < 200),  "\n")
  cat("  Cells with nFeature_RNA > 6000 :", sum(meta$nFeature_RNA > 6000), "\n")
  cat("  Cells with nCount_RNA > 30000  :", sum(meta$nCount_RNA > 30000),  "\n")
  cat("  Cells with percent.mt > 15     :", sum(meta$percent.mt > 15),     "\n")
}

# ── Apply filters ─────────────────────────────────────────────────────────
# Thresholds based on:
#   nFeature_RNA > 200   : removes empty droplets (Luecken & Theis 2019)
#   nFeature_RNA < 6000  : removes doublets; 6000 chosen (not 5000) to retain
#                          T/NK cells that co-exist with CLL B cells
#   nCount_RNA   < 30000 : removes high-UMI doublets not caught by gene count
#   percent.mt   < 15    : removes dying cells (Ilicic et al. 2016)
#
# UPDATE these values if your violin plots suggest different breakpoints.

qc_filter <- function(obj) {
  subset(obj,
         subset = nFeature_RNA > 200  &
                  nFeature_RNA < 6000 &
                  nCount_RNA   < 30000 &
                  percent.mt   < 15)
}

obj_MV1_filtered <- qc_filter(obj_MV1)
obj_MV2_filtered <- qc_filter(obj_MV2)

# ── Post-filter summary ───────────────────────────────────────────────────
cat("\n=== AFTER FILTERING ===\n\n")

for (chip_name in c("MV1", "MV2")) {
  before <- get(paste0("obj_", chip_name))
  after  <- get(paste0("obj_", chip_name, "_filtered"))
  n_removed  <- ncol(before) - ncol(after)
  pct_kept   <- round(ncol(after) / ncol(before) * 100, 1)
  
  cat(sprintf("Chip %s: %d → %d cells  (%d removed, %.1f%% kept)\n",
              chip_name, ncol(before), ncol(after), n_removed, pct_kept))
  cat("  Per sample_id after filter:\n")
  print(table(after$sample_id))
  cat("\n")
}

total_before <- ncol(obj_MV1) + ncol(obj_MV2)
total_after  <- ncol(obj_MV1_filtered) + ncol(obj_MV2_filtered)
cat(sprintf("TOTAL: %d → %d cells  (%d removed, %.1f%% kept)\n",
            total_before, total_after,
            total_before - total_after,
            total_after / total_before * 100))

# ── Overwrite with filtered objects ──────────────────────────────────────
# Only run this line once you are happy with the thresholds above.
obj_MV1 <- obj_MV1_filtered
obj_MV2 <- obj_MV2_filtered


## Section 4 — Normalization with scran

We use **scran** pooling-based size factor normalization instead of simple library-size scaling.

### Why scran for CLL?
- The CLL tumor clone dominates the sample (~80–96% of cells). Simple scaling would treat the mean expression of that dominant clone as the reference, distorting normalization for the minority T/NK/monocyte populations.
- scran first clusters cells, then pools them within clusters to estimate size factors robustly even for low-count cells.
- Normalization is done **per chip** independently, before merging.

In [ ]:
normalize_with_scran <- function(obj, chip_label) {
  message("Normalizing: ", chip_label)
  sce        <- as.SingleCellExperiment(obj)
  quick_cl   <- quickCluster(sce, min.size = 50)
  message("  Pooling clusters: ", nlevels(factor(quick_cl)))
  sce        <- computeSumFactors(sce, clusters = quick_cl)
  sf         <- sizeFactors(sce)
  message("  Size factors — min: ", round(min(sf),3),
          " | median: ", round(median(sf),3),
          " | max: ", round(max(sf),3))
  sce        <- logNormCounts(sce)
  obj <- SetAssayData(obj,
                       assay    = "RNA",
                       layer    = "data",
                       new.data = logcounts(sce))
  return(obj)
}

obj_MV1 <- normalize_with_scran(obj_MV1, "MV1")
obj_MV2 <- normalize_with_scran(obj_MV2, "MV2")

## Section 5 — Merge Chips

Merge MV1 and MV2 into one Seurat object.  
`add.cell.ids` prefixes each barcode with the chip name so there are no collisions if the same sequence appears in both chips.

In [ ]:
merged <- merge(
  x            = obj_MV1,
  y            = obj_MV2,
  add.cell.ids = c("MV1", "MV2"),
  merge.data   = TRUE   # Preserve scran-normalized RNA data layer
)

cat("Cells per biological sample:\n"); print(table(merged$sample_id))
cat("\nCells per chip:\n");           print(table(merged$chip))
cat("\nPatient x timepoint:\n");      print(table(merged$patient, merged$timepoint))
cat("\nCell types from demux file:\n"); print(table(merged$celltype_demux))

## Section 6 — Highly Variable Gene Selection

This step identifies **highly variable genes (HVGs)** in the merged object. HVGs are genes whose expression varies more than expected across cells after accounting for their average expression level. These genes are usually the most informative for downstream dimensionality reduction, clustering, and cell-state discovery.

`FindVariableFeatures()` stores the selected HVGs inside the Seurat object. Here we use `selection.method = "vst"`, Seurat's variance-stabilizing method. It models the mean–variance relationship and prioritizes genes with unusually high standardized variance, instead of simply selecting genes with high raw expression or high technical noise.

We keep `nfeatures = 3000`, which is a reasonable choice for this dataset because it preserves enough biological signal for heterogeneous CLL blood samples while excluding many low-information genes. These selected genes will be used in the next steps for scaling and PCA.

The plot shows each gene according to its average expression and standardized variance. The labeled genes are the top 20 HVGs. These labels are useful for a quick sanity check: expected immune, activation, cell-cycle, mitochondrial, ribosomal, or immunoglobulin-related genes may appear depending on the biology and sample composition. If the top genes are dominated by technical artifacts, this would suggest revisiting QC or filtering before PCA.

In [ ]:
merged <- FindVariableFeatures(merged,
                               selection.method = "vst",
                               nfeatures = 3000)

top20 <- head(VariableFeatures(merged), 20)
LabelPoints(plot   = VariableFeaturePlot(merged),
            points = top20,
            repel  = TRUE) +
  ggtitle("Top variable genes")

## Section 7 — Scaling, Cell Cycle Scoring, and PCA

**Scaling** centers each gene to mean = 0, sd = 1, so no gene dominates PCA due to high absolute expression.  
**Cell cycle scoring** assigns each cell a G1/S/G2M phase — useful to decide whether to regress it out.  
**PCA** is run on the HVGs. The elbow plot shows how many PCs capture meaningful variance; typically 15–25 for CLL blood.

In [ ]:
# Cell cycle scoring
s.genes   <- cc.genes$s.genes
g2m.genes <- cc.genes$g2m.genes
merged    <- CellCycleScoring(merged,
                              s.features   = s.genes,
                              g2m.features = g2m.genes,
                              set.ident    = FALSE)

# Scale — regress out mitochondrial fraction
# Add "S.Score", "G2M.Score" to vars.to.regress if cell cycle confounds clustering
merged <- ScaleData(merged,
                    vars.to.regress = c("percent.mt"),
                    features        = VariableFeatures(merged))

# PCA
merged <- RunPCA(merged,
                 features = VariableFeatures(merged),
                 npcs     = 50,
                 verbose  = FALSE)

message("PCA complete.")

In [ ]:
# Elbow plot — choose n_pcs where the curve flattens
ElbowPlot(merged, ndims = 50) +
  ggtitle("Elbow plot: variance explained per PC")

# UPDATE this value based on the elbow plot before running Section 8
n_pcs <- 25

## Section 8 — Batch Correction with Harmony

### Why Harmony here?

You have **two confounded sources of variation**:

| Source | Variable | Values |
|--------|----------|--------|
| Technical | Chip | MV1, MV2 |
| Biological | Patient | P457, P477 |

These are partially confounded: P457-Dx and P477-Ref are both on MV1; P477-Dx and P457-Ref are both on MV2.  
Harmony corrects the **PCA embedding** without modifying the count matrix. It iteratively adjusts PC coordinates to remove group-level offsets. We correct for both `patient` and `chip` simultaneously.

> The corrected embedding is stored in the `"harmony"` reduction and is used for all downstream steps (UMAP, clustering).

In [ ]:
merged <- RunHarmony(
  merged,
  group.by.vars = c("patient", "chip"),  # Correct inter-patient AND inter-chip
  dims.use      = 1:n_pcs,
  assay.use     = "RNA",
  verbose       = TRUE
)

message("Harmony complete. Corrected embedding stored as 'harmony'.")

## Section 9 — UMAP and Clustering

UMAP projects cells into 2D using the Harmony-corrected PCs.  
Clustering uses the Louvain algorithm at multiple resolutions — inspect each before committing.  
For CLL blood: resolution 0.5 is a good starting point.

In [ ]:
merged <- RunUMAP(merged, reduction = "harmony", dims = 1:n_pcs, seed.use = 42)
merged <- FindNeighbors(merged, reduction = "harmony", dims = 1:n_pcs)
merged <- FindClusters(merged, resolution = c(0.3, 0.5, 0.8, 1.0), verbose = FALSE)

message("UMAP and clustering complete.")

In [ ]:
# Compare resolutions
p03 <- DimPlot(merged, group.by = "RNA_snn_res.0.3", label = TRUE, repel = TRUE) + ggtitle("res 0.3")
p05 <- DimPlot(merged, group.by = "RNA_snn_res.0.5", label = TRUE, repel = TRUE) + ggtitle("res 0.5")
p08 <- DimPlot(merged, group.by = "RNA_snn_res.0.8", label = TRUE, repel = TRUE) + ggtitle("res 0.8")
(p03 | p05 | p08) + plot_annotation(title = "Clustering at different resolutions")

In [ ]:
# Check batch correction: after Harmony, chips/patients should be mixed in UMAP
p_patient  <- DimPlot(merged, group.by = "patient",   label = FALSE) + ggtitle("By patient")
p_chip     <- DimPlot(merged, group.by = "chip",      label = FALSE) + ggtitle("By chip")
p_time     <- DimPlot(merged, group.by = "timepoint", label = FALSE) + ggtitle("By timepoint")
p_phase    <- DimPlot(merged, group.by = "Phase",     label = FALSE) + ggtitle("Cell cycle phase")
(p_patient | p_chip) / (p_time | p_phase)

In [ ]:
# Set the working resolution (update after inspecting plots above)
Idents(merged) <- "RNA_snn_res.0.5"
message("Active identity set to resolution 0.5")

## Section 10 — Automated Cell Type Annotation (SingleR)

SingleR compares each cell's log-normalized profile against bulk RNA-seq reference profiles of known cell types.  
We use the **Monaco 2019 immune reference** — the best available for blood / PBMC because it includes fine-grained immune subtypes relevant to CLL microenvironment (plasmacytoid DCs, MAIT cells, exhausted T cells, regulatory T cells).

> **Bonus:** Since you already have `celltype_demux` from your labeling file, we compare SingleR's automated annotation against it — a useful sanity check.

In [ ]:
ref_monaco    <- celldex::MonacoImmuneData()
sce_singler   <- as.SingleCellExperiment(merged)

singler_results <- SingleR(
  test    = sce_singler,
  ref     = ref_monaco,
  labels  = ref_monaco$label.main,   # Use label.fine for more granularity
  BPPARAM = MulticoreParam(4)
)

merged$singler_label  <- singler_results$labels
merged$singler_pruned <- singler_results$pruned.labels  # NAs = low confidence

message("SingleR annotation complete.")

In [ ]:
DimPlot(merged, reduction = "umap", group.by = "singler_label",
        label = TRUE, repel = TRUE, pt.size = 0.5) +
  ggtitle("SingleR annotation (Monaco reference)") +
  theme_classic()

In [ ]:
# Annotation confidence heatmap
# Rows = clusters, columns = reference cell types
# High score in one column = confident assignment
plotScoreHeatmap(singler_results,
                 clusters    = merged$RNA_snn_res.0.5,
                 show.pruned = TRUE)

In [ ]:
# Cross-tabulate SingleR labels vs your existing demux cell types
cat("SingleR vs demux cell type comparison:\n")
print(table(SingleR = merged$singler_label,
            Demux   = merged$celltype_demux))

## Section 11 — Manual Annotation with Canonical Markers

Validate and refine SingleR with canonical marker genes.  
CLL peripheral blood contains: B cells (dominant tumor clone), CD4 T cells, CD8 T cells, NK cells, monocytes.  
The CLL immunophenotype is **CD19⁺ CD5⁺ CD23⁺** — co-expression of CD5 on B cells is the diagnostic hallmark.

In [ ]:
canonical_markers <- list(
  "B_general"     = c("CD19", "MS4A1", "CD79A", "CD79B"),
  "CLL_tumor"     = c("CD5", "CD23", "CD200", "FCER2"),    # CLL hallmark co-expression
  "AID_expressing"= c("AICDA"),                             # KEY gene for your study
  "Plasma_cell"   = c("IGHG1", "IGHA1", "SDC1", "PRDM1"),
  "T_CD4"         = c("CD3D", "CD3E", "CD4", "IL7R"),
  "T_CD8"         = c("CD3D", "CD8A", "CD8B", "GZMB"),
  "T_reg"         = c("FOXP3", "IL2RA", "CTLA4"),
  "T_exhausted"   = c("PDCD1", "LAG3", "HAVCR2"),
  "NK"            = c("NCAM1", "NKG7", "GNLY", "KLRD1"),
  "Monocyte"      = c("CD14", "LYZ", "S100A8", "S100A9"),
  "Dendritic"     = c("CLEC4C", "LILRA4", "IL3RA"),
  "Proliferating" = c("MKI67", "TOP2A", "PCNA", "CCNB1"),
  "AID_pathway"   = c("AICDA", "UNG", "MSH2", "MSH6", "EXO1", "POLH")
)

# Dot plot: percent expressing + mean expression per cluster
DotPlot(merged,
        features = unique(unlist(canonical_markers)),
        group.by = "RNA_snn_res.0.5",
        dot.scale = 6) +
  RotatedAxis() +
  scale_color_viridis(option = "plasma") +
  ggtitle("Canonical markers by cluster")

In [ ]:
# Feature plots for the four most diagnostic genes
FeaturePlot(merged,
            features  = c("AICDA", "CD19", "CD5", "MKI67"),
            reduction = "umap",
            ncol      = 2,
            order     = TRUE,
            min.cutoff = "q05") &
  scale_color_viridis(option = "magma")

## Section 12 — Cluster Annotation and Tumor Cell Identification

> **Action required:** Fill in the `cluster_annotation` vector below **after** reviewing the dot plot and feature plots in Section 11.  
> The cluster numbers here are placeholders — match them to what you see in your data.

In [ ]:
# TEMPLATE — adjust cluster numbers after inspecting Section 11 plots
cluster_annotation <- c(
  "0" = "CLL_B_cell",        # CD19+ CD5+ CD23+ (tumor bulk)
  "1" = "CLL_B_cell",
  "2" = "T_CD4",
  "3" = "T_CD8",
  "4" = "NK",
  "5" = "Monocyte",
  "6" = "CLL_AID_positive",  # Cluster with detectable AICDA
  "7" = "Proliferating_B",   # MKI67+ B cells
  "8" = "T_reg"
)

merged$cell_type <- recode(as.character(merged$RNA_snn_res.0.5),
                           !!!cluster_annotation)

# Tumor cell flag
merged$is_tumor <- merged$cell_type %in%
  c("CLL_B_cell", "CLL_AID_positive", "Proliferating_B")

# AID-positive flag (AICDA transcript detected at any level)
aicda_expr       <- FetchData(merged, vars = "AICDA")
merged$is_AID_positive <- aicda_expr$AICDA > 0

message("AID+ cells: ", sum(merged$is_AID_positive),
        " (", round(mean(merged$is_AID_positive)*100, 2), "% of all cells)")

DimPlot(merged, reduction = "umap", group.by = "cell_type",
        label = TRUE, repel = TRUE, pt.size = 0.4) +
  ggtitle("Cell type annotation") + theme_classic()

## Section 13 — Proliferation Scoring

In [ ]:
# Combined proliferation score (S + G2M)
merged$prolif_score <- merged$S.Score + merged$G2M.Score

VlnPlot(merged, features = "prolif_score",
        group.by = "cell_type", pt.size = 0) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  ggtitle("Proliferation score by cell type")

In [ ]:
# Tumor cells only: proliferation across time points
tumor_cells <- subset(merged, subset = is_tumor == TRUE)
VlnPlot(tumor_cells,
        features  = c("S.Score", "G2M.Score", "MKI67"),
        group.by  = "timepoint",
        ncol      = 3,
        pt.size   = 0) +
  plot_annotation(title = "Proliferation: Diagnosis vs. Refractory (tumor cells)")

## Section 14 — Pathway Enrichment (GSEA with fgsea)

Gene set enrichment analysis ranks all expressed genes by fold change between groups, then tests whether curated gene sets are concentrated at the top or bottom of that ranking.

**Gene sets used:**
- **Hallmark (MSigDB H):** 50 well-curated canonical pathways — includes IFN-α, IFN-γ, KRAS, MYC, apoptosis
- **KEGG (MSigDB C2):** Detailed metabolic and signaling pathways

**Three comparisons:**
1. Diagnosis vs. Refractory in tumor B cells
2. AID-positive vs. AID-negative tumor cells
3. Per-patient separately (P457 and P477) — checks concordance

In [ ]:
# Load gene sets from MSigDB
hallmark_sets <- msigdbr(species = "Homo sapiens", category = "H") %>%
  split(x = .$gene_symbol, f = .$gs_name)

kegg_sets <- msigdbr(species = "Homo sapiens", category = "C2",
                     subcategory = "CP:KEGG") %>%
  split(x = .$gene_symbol, f = .$gs_name)

message("Hallmark pathways: ", length(hallmark_sets))
message("KEGG pathways: ",     length(kegg_sets))

In [ ]:
# Helper function: compute log-fold-change ranking then run fgsea
run_gsea_comparison <- function(seurat_obj, group_col, group_a, group_b,
                                gene_sets, label = "") {
  message("\nGSEA: ", label)
  cells_a <- WhichCells(seurat_obj, expression = !!sym(group_col) == group_a)
  cells_b <- WhichCells(seurat_obj, expression = !!sym(group_col) == group_b)
  
  if (length(cells_a) < 10 | length(cells_b) < 10) {
    warning("Too few cells — skipping: ", label); return(NULL)
  }
  
  mat    <- GetAssayData(seurat_obj, slot = "data")
  lfc    <- rowMeans(mat[, cells_a, drop=FALSE]) - rowMeans(mat[, cells_b, drop=FALSE])
  lfc    <- sort(lfc[!is.na(lfc) & is.finite(lfc)], decreasing = TRUE)
  
  set.seed(42)
  result <- fgsea(pathways = gene_sets, stats = lfc,
                  minSize = 15, maxSize = 500, eps = 0)
  result$comparison <- label
  result$group_a    <- group_a
  result$group_b    <- group_b
  return(result[order(result$pval), ])
}

In [ ]:
# Comparison A: Diagnosis vs. Refractory in tumor cells
tumor_obj <- subset(merged, subset = is_tumor == TRUE)

gsea_dx_vs_ref_hallmark <- run_gsea_comparison(
  tumor_obj, "timepoint", "Diagnosis", "Refractory",
  hallmark_sets, "Diagnosis_vs_Refractory_Hallmark")

gsea_dx_vs_ref_kegg <- run_gsea_comparison(
  tumor_obj, "timepoint", "Diagnosis", "Refractory",
  kegg_sets, "Diagnosis_vs_Refractory_KEGG")

In [ ]:
# Comparison B: AID+ vs AID- tumor cells
gsea_AID_hallmark <- run_gsea_comparison(
  tumor_obj, "is_AID_positive", "TRUE", "FALSE",
  hallmark_sets, "AID_pos_vs_AID_neg_Hallmark")

In [ ]:
# Comparison C: Per-patient (check concordance between P457 and P477)
gsea_P457 <- run_gsea_comparison(
  subset(tumor_obj, subset = patient == "P457"),
  "timepoint", "Diagnosis", "Refractory",
  hallmark_sets, "P457_Dx_vs_Ref")

gsea_P477 <- run_gsea_comparison(
  subset(tumor_obj, subset = patient == "P477"),
  "timepoint", "Diagnosis", "Refractory",
  hallmark_sets, "P477_Dx_vs_Ref")

## Section 15 — Per-Cell Pathway Scoring (IFN-α, IFN-γ, KRAS)

`AddModuleScore()` assigns each individual cell a score for your pathways of interest, normalized against randomly sampled gene sets of the same size.  
This lets you overlay pathway activity directly on UMAP and identify which subpopulation (AID+, proliferating, etc.) is enriched.

In [ ]:
hallmark_df <- msigdbr(species = "Homo sapiens", category = "H")
get_genes   <- function(name) {
  hallmark_df %>% filter(gs_name == name) %>% pull(gene_symbol) %>% unique() %>% list()
}

merged <- AddModuleScore(merged, get_genes("HALLMARK_INTERFERON_ALPHA_RESPONSE"), name="IFNa_score")
merged <- AddModuleScore(merged, get_genes("HALLMARK_INTERFERON_GAMMA_RESPONSE"),  name="IFNg_score")
merged <- AddModuleScore(merged, get_genes("HALLMARK_KRAS_SIGNALING_UP"),          name="KRAS_score")
merged <- AddModuleScore(merged, get_genes("HALLMARK_MYC_TARGETS_V1"),             name="MYC_score")
merged <- AddModuleScore(merged, get_genes("HALLMARK_APOPTOSIS"),                  name="Apoptosis_score")
merged <- AddModuleScore(merged, get_genes("HALLMARK_G2M_CHECKPOINT"),             name="G2M_score")

message("Module scores added. Note: Seurat appends '1' to each name.")

In [ ]:
# Pathway scores on UMAP
FeaturePlot(merged,
            features   = c("IFNa_score1","IFNg_score1","KRAS_score1","MYC_score1"),
            reduction  = "umap",
            ncol       = 2,
            order      = TRUE,
            min.cutoff = "q05",
            max.cutoff = "q95") &
  scale_color_viridis(option = "magma") &
  theme_classic()

In [ ]:
tumor_obj <- subset(merged, subset = is_tumor == TRUE)

# Diagnosis vs Refractory
VlnPlot(tumor_obj,
        features = c("IFNa_score1","IFNg_score1","KRAS_score1"),
        group.by = "timepoint", ncol = 3, pt.size = 0) +
  plot_annotation(title = "Pathway scores: Diagnosis vs. Refractory (tumor cells)")

In [ ]:
# AID+ vs AID-
VlnPlot(tumor_obj,
        features = c("IFNa_score1","IFNg_score1","KRAS_score1"),
        group.by = "is_AID_positive", ncol = 3, pt.size = 0) +
  plot_annotation(title = "Pathway scores: AID+ vs AID- tumor cells")

## Section 16 — Visualize GSEA Results

In [ ]:
plot_gsea_bubble <- function(gsea_result, n_top = 20, title = "") {
  if (is.null(gsea_result)) { message("No result to plot."); return(invisible(NULL)) }
  df <- gsea_result %>%
    filter(!is.na(padj)) %>%
    arrange(padj) %>%
    head(n_top) %>%
    mutate(
      pathway_short = str_replace(pathway, "HALLMARK_|KEGG_", ""),
      direction     = ifelse(NES > 0, "Enriched in Group A", "Enriched in Group B")
    )
  ggplot(df, aes(x = NES, y = reorder(pathway_short, NES),
                 size = -log10(padj + 1e-10), color = direction)) +
    geom_point(alpha = 0.8) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "gray50") +
    scale_color_manual(values = c("Enriched in Group A" = "#E64B35",
                                  "Enriched in Group B" = "#4DBBD5")) +
    scale_size_continuous(range = c(2, 8), name = "-log10(FDR)") +
    labs(title = title, x = "NES", y = NULL, color = "Direction") +
    theme_classic(base_size = 11)
}

plot_gsea_bubble(gsea_dx_vs_ref_hallmark,
                 title = "Hallmark GSEA: Diagnosis vs. Refractory (tumor B cells)")

In [ ]:
plot_gsea_bubble(gsea_AID_hallmark,
                 title = "Hallmark GSEA: AID+ vs. AID- tumor cells")

In [ ]:
# Concordance: do P457 and P477 show the same pathway enrichments?
if (!is.null(gsea_P457) & !is.null(gsea_P477)) {
  concordance <- merge(
    gsea_P457[, c("pathway","NES","padj")],
    gsea_P477[, c("pathway","NES","padj")],
    by = "pathway", suffixes = c("_P457","_P477")
  )
  ggplot(concordance, aes(x = NES_P457, y = NES_P477, label = pathway)) +
    geom_point(aes(color = padj_P457 < 0.05 | padj_P477 < 0.05), size = 2) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
    geom_text_repel(
      data = concordance %>% filter(padj_P457 < 0.05 | padj_P477 < 0.05),
      aes(label = str_replace(pathway, "HALLMARK_", "")),
      size = 3, max.overlaps = 15
    ) +
    scale_color_manual(values = c("FALSE"="gray70","TRUE"="#E64B35"),
                       name = "FDR < 0.05 in either patient") +
    theme_classic() +
    ggtitle("NES concordance: P457 vs P477 (Dx vs Ref)")
}

## Section 17 — Pseudobulk Differential Expression (DESeq2)

For formal DE, **pseudobulk** is the gold standard. We collapse all cells from each biological sample into a single count profile, then run DESeq2 on the 4 sample-level profiles.

This avoids **inflated degrees of freedom** from treating thousands of cells as independent replicates.  
The design `~ patient + timepoint` controls for inter-patient variation — critical with only 2 patients.

In [ ]:
run_pseudobulk_DE <- function(seurat_obj, cell_subset,
                               comparison_col, group_a, group_b,
                               sample_col = "sample_id") {
  obj_sub   <- subset(seurat_obj, subset = cell_type %in% cell_subset)
  counts_pb <- AggregateExpression(obj_sub, assays = "RNA", slot = "counts",
                                   group.by = sample_col, return.seurat = FALSE)$RNA
  meta_df   <- seurat_obj@meta.data %>%
    select(all_of(c(sample_col, comparison_col, "patient", "chip"))) %>%
    distinct() %>%
    filter(!!sym(comparison_col) %in% c(group_a, group_b)) %>%
    column_to_rownames(sample_col)
  
  common        <- intersect(colnames(counts_pb), rownames(meta_df))
  counts_pb     <- counts_pb[, common]
  meta_df       <- meta_df[common, , drop = FALSE]
  meta_df[[comparison_col]] <- factor(meta_df[[comparison_col]],
                                       levels = c(group_b, group_a))
  
  dds <- DESeqDataSetFromMatrix(
    countData = round(counts_pb),
    colData   = meta_df,
    design    = as.formula(paste0("~ patient + ", comparison_col))
  )
  dds <- DESeq(dds, test = "LRT", reduced = as.formula("~ patient"))
  res <- results(dds, contrast = c(comparison_col, group_a, group_b), alpha = 0.05) %>%
    as.data.frame() %>% rownames_to_column("gene") %>% arrange(padj)
  
  return(list(dds = dds, results = res))
}

In [ ]:
de_dx_vs_ref <- run_pseudobulk_DE(
  seurat_obj     = merged,
  cell_subset    = c("CLL_B_cell","CLL_AID_positive","Proliferating_B"),
  comparison_col = "timepoint",
  group_a        = "Diagnosis",
  group_b        = "Refractory"
)

cat("Top DE genes (Diagnosis vs Refractory, tumor cells):\n")
print(head(de_dx_vs_ref$results %>% filter(padj < 0.05), 20))

In [ ]:
# Volcano plot
de_res <- de_dx_vs_ref$results %>%
  mutate(sig = case_when(
    padj < 0.05 & log2FoldChange >  1 ~ "Up in Dx",
    padj < 0.05 & log2FoldChange < -1 ~ "Up in Ref",
    TRUE ~ "NS"))

ggplot(de_res, aes(x = log2FoldChange, y = -log10(padj + 1e-300), color = sig)) +
  geom_point(alpha = 0.6, size = 0.8) +
  scale_color_manual(values = c("Up in Dx"="#E64B35","Up in Ref"="#4DBBD5","NS"="gray70")) +
  geom_vline(xintercept = c(-1,1), linetype="dashed") +
  geom_hline(yintercept = -log10(0.05), linetype="dashed") +
  geom_text_repel(
    data = de_res %>% filter(padj < 0.01 & abs(log2FoldChange) > 2),
    aes(label = gene), size = 3, max.overlaps = 20
  ) +
  theme_classic() +
  ggtitle("DESeq2 pseudobulk: Diagnosis vs. Refractory (tumor B cells)")

## Section 18 — Save All Outputs

In [ ]:
# Full annotated Seurat object
saveRDS(merged, "CLL_scRNAseq_merged_annotated.rds")

# GSEA tables
if (!is.null(gsea_dx_vs_ref_hallmark))
  write.csv(gsea_dx_vs_ref_hallmark, "GSEA_Hallmark_Dx_vs_Ref_tumor.csv",     row.names=FALSE)
if (!is.null(gsea_AID_hallmark))
  write.csv(gsea_AID_hallmark,       "GSEA_Hallmark_AID_pos_vs_neg.csv",      row.names=FALSE)
if (!is.null(gsea_dx_vs_ref_kegg))
  write.csv(gsea_dx_vs_ref_kegg,     "GSEA_KEGG_Dx_vs_Ref_tumor.csv",         row.names=FALSE)

# DESeq2 results
write.csv(de_dx_vs_ref$results, "DESeq2_pseudobulk_Dx_vs_Ref_tumor.csv", row.names=FALSE)

# Per-cell pathway scores
module_df <- FetchData(merged,
  vars = c("cell_type","patient","timepoint","is_AID_positive",
           "IFNa_score1","IFNg_score1","KRAS_score1","MYC_score1",
           "Apoptosis_score1","G2M_score1","prolif_score"))
write.csv(module_df, "pathway_scores_per_cell.csv")

# Reproducibility
writeLines(capture.output(sessionInfo()), "sessionInfo.txt")

message("\n=== Pipeline complete ===")
message("Saved files:")
for (f in c("CLL_scRNAseq_merged_annotated.rds",
            "GSEA_Hallmark_Dx_vs_Ref_tumor.csv",
            "GSEA_Hallmark_AID_pos_vs_neg.csv",
            "GSEA_KEGG_Dx_vs_Ref_tumor.csv",
            "DESeq2_pseudobulk_Dx_vs_Ref_tumor.csv",
            "pathway_scores_per_cell.csv",
            "sessionInfo.txt")) message("  ", f)